## XML Sources

In [4]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup
import pandas as pd
import re

def retrieve_xml_articles(xml_links):

    driver = webdriver.Chrome()

    # Initalizing news articles df  
    new_articles = pd.DataFrame(columns=["title", "link", "source", "language", "published_date", "last_modified", "keywords"])  

    for url in xml_links:
        driver.get(url)
        xml_site = driver.page_source
        soup = BeautifulSoup(xml_site, "lxml-xml")

        articles = soup.find_all("url")

        for article in articles:

            title = article.find("news:title").get_text()
            link = article.find("loc").text # get_text doesn't work here for links 
            source = article.find("news:name").get_text()
            language = article.find("news:language").get_text()
            published_date = article.find("news:publication_date").get_text()
            last_modified = article.find("lastmod").get_text(strip=True) if article.find("lastmod") else None  
            keywords = article.find("news:keywords").get_text(strip=True) if article.find("news:keywords") else None


            tempdf = pd.DataFrame({
            "title": title,
            "link": link,
            "source": source,
            "language": language,
            "published_date": published_date,
            "last_modified": last_modified,
            "keywords": [keywords]}, index=[0]) 

            new_articles = pd.concat([new_articles, tempdf], ignore_index=True)

    driver.quit()

    # Use .replace when replacing entire value, else use str.replace which incoroprates regex
    new_articles["language"] = new_articles["language"].replace("en", "eng") 

    new_articles = new_articles[new_articles['language'] == 'eng']
    
    return new_articles

In [5]:
from sqlalchemy import text, create_engine

engine = create_engine("sqlite:///articles.db")

with engine.connect() as connection:
    xml_links = pd.read_sql_query(
        text("""SELECT article_links FROM article_sources" \
        "where source_type == 'XML'""" ), connection)
    
xml_links_list = xml_links['article_links'].tolist()

new_articles = retrieve_xml_articles(xml_links_list)

## RSS Feeds

In [6]:
import feedparser
import pandas as pd

def retrieve_rss_feeds(rss_urls_list):
    rss_stories = pd.DataFrame(columns=["title", "summary", "link", "published_date", "source"])

    for rss_url in rss_urls_list:

        # Parse the feed
        feed = feedparser.parse(rss_url)

        for entry in feed.entries: # Figure out way to limit entires

            tempdf = pd.DataFrame({
            "title": entry.title,
            "link": entry.link,
            "summary": entry.description,
            "published_date": entry.published,
            "source": feed.channel.title}, index=[0]) 
            
            rss_stories = pd.concat([rss_stories, tempdf], ignore_index=True)

    return rss_stories

In [7]:
from sqlalchemy import text, create_engine

engine = create_engine("sqlite:///articles.db")

with engine.connect() as connection:
    rss_links = pd.read_sql_query(
        text("""SELECT article_links FROM article_sources" \
        "where source_type == 'RSS'""" ), connection)
    
rss_links_list = rss_links['article_links'].tolist()

rss_stories = retrieve_rss_feeds(rss_links_list)

#### Next item is to build out categoies for rss feeds (Also better define categoies which is listed in {category_keywords})

## Concatenate RSS and XML Results

In [8]:
import pandas as pd
from dateutil import parser

final_output = pd.concat([new_articles, rss_stories], ignore_index=True)

# Defining how to normalize dates in a really simple way
def normalize_date(x):
    return parser.parse(x).astimezone(tz=None)

final_output['published_date'] = final_output['published_date'].apply(normalize_date)

# forcing everything into UTC for consistency
final_output['published_date'] = pd.to_datetime(final_output['published_date'], errors='coerce').dt.tz_convert('America/New_York')

In [ ]:
# Extracting keywords for specific articles using the links variable 
def extract_keywords(row):
    source = row['source']
    link = row['link']

    # Associated Press
    if source == "Associated Press":
        if "article/" in link:
            keywords = ", ".join(link.split("article/")[1].split("-")[:-1])
            return keywords
        else:
            return None

    # WSJ sources
    elif source in ["WSJ.com: World News", "Technology - WSJ.com"]:
        keywords = ", ".join(link.split("/")[-1].split("-")[:-1])
        return keywords

    # All other sources
    else:
        return row['keywords']


# Apply the function
final_output['keywords'] = final_output.apply(extract_keywords, axis=1)

In [ ]:
# Mapping a cateogry to each article based on 
import re

def extract_categories():

    category_keywords = {
        "China": ["china", "beijing", "xi", "taiwan"],
        "Semiconductors": ["semiconductor", "chip", "nvidia", "intel", "amd"],
        "Technology": ["ai", "data", "cloud", "software", "technology"],
        "Politics": ["election", "government", "policy", "congress"],
        "Finance": ["stock", "market", "finance", "economy", "inflation"]}

    final_output['category'] = None

    for i, row in final_output.iterrows():

        # Safely build combined text (handle NaN)
        title = '' if pd.isna(row.get('title')) else str(row['title'])
        summary = '' if pd.isna(row.get('summary')) else str(row['summary'])
        keywords = '' if pd.isna(row.get('keywords')) else str(row['keywords'])
        combined_text = " ".join([title, keywords, summary]).lower()

        # Initialize matched_category state to none
        matched_category = None

        for category, kw_list in category_keywords.items():
            if any(re.search(rf"\b{kw}\b", combined_text) for kw in kw_list):
                matched_category = category
                final_output.at[i, 'category'] = matched_category  # Use .at instead of loc or iloc
                break # exiting the loop on first match

        # Default fallback
        if not matched_category:
            final_output.at[i, 'category'] = "Other"

    return final_output

In [1]:
import pandas as pd
final_output = pd.read_excel("final_news_output.xlsx")

In [2]:
# manually input a separate category for powerbi 
final_output.loc[final_output['source'] == "Microsoft Power BI Blog | Microsoft Power BI", 'category'] = "PowerBi"

# Filtering out all news articles that didn't get assigned a caetegory
final_output = final_output[final_output['category'] != 'Other']

# Capitalizes all the keywords that are in list format
final_output['keywords'] = final_output['keywords'].apply(
    lambda x: ", ".join(w.strip().capitalize() for w in x.split(",")) if isinstance(x, str) else x)

## Inital creation articles DB in SQL

In [8]:
from sqlalchemy import create_engine, text

engine = create_engine("sqlite:///articles.db")

with engine.begin() as conn:

        # habits db creation
        conn.execute(text("""CREATE TABLE news_articles (
                            id INTEGER PRIMARY KEY AUTOINCREMENT,
                            title TEXT NOT NULL,
                            link TEXT,
                            source TEXT,
                            language TEXT,
                            published_date DATETIME,
                            last_modified DATETIME,
                            keywords TEXT,
                            summary TEXT,
                            category TEXT,
                            upload_date DATETIME NOT NULL);"""))

In [9]:
# Appending the new raw data to my sql lite table
from datetime import datetime
import pytz

final_output = final_output

eastern = pytz.timezone('America/New_York')
local_time = datetime.now(eastern).strftime('%Y-%m-%d %H:%M:%S')

final_output["upload_date"] = local_time

final_output.to_sql(name="news_articles", con=engine, if_exists='append', index=False)

128

## Inserting new data into SQL DB

#### Think about a new way to upload new observations, maybe like bring both down merge them based on limited columns. filter new articles df to those that don't exist in db then use .to_sql function to upload new info

In [ ]:
# Reading in article table from sql db
from sqlalchemy import create_engine, text
from datetime import datetime
import pytz

eastern = pytz.timezone('America/New_York')
local_time = datetime.now(eastern).strftime('%Y-%m-%d %H:%M:%S')

final_output['upload_date'] = local_time

engine = create_engine("sqlite:///articles.db")

with engine.connect() as connection:
    articles_db = pd.read_sql_query(
        text("SELECT * FROM news_articles"),connection,
        parse_dates=['published_date', 'last_modified', 'upload_date'])

articles_db = articles_db.drop(columns=["id"])

# Using the link variable to check for duplicates
combined = pd.concat([articles_db, final_output]).drop_duplicates(subset="link")

# Filtering the combined data to only include this batches articles that are not duplicates
combined = combined[combined['upload_date'] == local_time]

In [15]:
combined.to_sql(name="news_articles", con=engine, if_exists='append', index=False)

91

## Inserting new article links into article_sources SQL DB

In [19]:
# Add 1 link at a time
from sqlalchemy import text, create_engine

engine = create_engine("sqlite:///articles.db")

source = "XML"
link = "https://googlecrawl.npr.org/news/sitemap_news.xml"

with engine.begin() as conn:
    # Insert new article
    conn.execute(
        text("""
            INSERT INTO article_sources 
            (source_type, article_links)
            VALUES (:source_type, :article_links)"""),
        {
            "source_type": source,
            "article_links": link,
        },
    )

In [ ]:
# Add a list of links at a time
from sqlalchemy import text, create_engine

engine = create_engine("sqlite:///articles.db")

source = "XML"

xml_articles = ["https://apnews.com/news-sitemap-content.xml", "https://googlecrawl.npr.org/news/sitemap_news.xml"]
rss_urls = ["https://powerbi.microsoft.com/en-us/blog/feed/", "https://feeds.content.dowjones.io/public/rss/RSSWorldNews", "https://feeds.content.dowjones.io/public/rss/RSSWSJD"]

with engine.begin() as conn:

    for link in xml_articles:
        # Insert new article
        conn.execute(
            text("""
                INSERT INTO article_sources 
                (source_type, article_links)
                VALUES (:source_type, :article_links)"""),
            {
                "source_type": source,
                "article_links": link,
            },
        )

## Inserting new email recipients into article_sources SQL DB

In [21]:
# Add 1 link at a time
from sqlalchemy import text, create_engine

engine = create_engine("sqlite:///articles.db")

email = "iacovone3@icloud.com"

with engine.begin() as conn:
    # Insert new article
    conn.execute(
        text("""
            INSERT INTO email_recipients 
            (emails)
            VALUES (:emails)"""),
        {
            "emails": email
        },
    )

## Converting the df into a markdown file for export

In [ ]:
import pandas as pd
from sqlalchemy import text

# Grabbing email recipients dynamically from SQL
with engine.connect() as connection:
    email_recipients = pd.read_sql_query(
        text("""SELECT emails FROM email_recipients""" ), connection)

In [ ]:
import pandas as pd
from datetime import datetime
from pathlib import Path
import random

def df_to_markdown(df):

    # Grabbing unique categories from sqlite db
    categories = df['category'].unique().tolist() 

    # Build markdown string
    markdown_output = f"# 📰 Demolition Daily\n\n"

    # list of emojis to itterate through for fun!
    news_emojis = ["🆕", "📰", "🌎", "📢", "🖥️", "🗃️", "🗞️"] 

    for category in categories:
        subset = df[df["category"] == category]
        if subset.empty:
            continue

        markdown_output += f"""<h2 style="text-decoration: underline; font-size: 28px; margin-top: 30px;">{category}</h2> \n\n"""

        for i, row in subset.iterrows():
            # Handle summary / keywords fallback
            summary_text = None
            if pd.notna(row.get('summary')) and str(row['summary']).strip():
                summary_text = row['summary']
            elif pd.notna(row.get('keywords')) and str(row['keywords']).strip():
                summary_text = row['keywords']
            else:
                summary_text = 'No summary available.'

            # Handle source field gracefully
            source = row.get('source', 'Unknown Source')

            # Build the visually grouped “Card” markdown block
            markdown_output += (
                f"> ### {random.choice(news_emojis)} [{row['title']}]({row['link']})\n"
                f"> **Source:** {source}  \n"
                f"> **Published:** {row['published_date']}  \n"
                f"> **Tags:** {summary_text}\n\n"
            )

    # Write to markdown file
    output_path = Path("news_digest.md")
    output_path.write_text(markdown_output, encoding="utf-8")

    print(f"✅ Markdown file generated at: {output_path.resolve()}")

df_to_markdown(final_output)

✅ Markdown file generated at: C:\Users\ryani\Coding Projects\News Aggregator\news_digest.md


## Exporting the news via email

In [ ]:
# Version 1 converting markdown to html and sending that over email  
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
import markdown
import pandas as pd
from dotenv import load_dotenv
import os
from sqlalchemy import create_engine, text

engine = create_engine("sqlite:///articles.db")

# Load variables from .env file
load_dotenv()

email_pass = os.getenv("email_pass")
email_sender = os.getenv("email_sender")

# Grabbing email recipients dynamically from SQL
with engine.connect() as connection:
    email_recipients = pd.read_sql_query(
        text("""SELECT emails FROM email_recipients""" ), connection)
    
email_recipients = email_recipients['emails'].tolist()

# Grabbing today's date as a string for the subject
today_date = datetime.now().strftime("%A, %B %d, %Y")

def send_markdown_as_email(markdown_path, email_sender, email_pass, email_to):
    with open(markdown_path, "r", encoding="utf-8") as f:
        md_text = f.read()

    # Converting markdown file to HTML (actually works pretty well!)
    html = markdown.markdown(md_text)

    msg = MIMEMultipart("alternative")
    msg["From"] = email_sender
    msg["To"] = ", ".join(email_to)  # Convert list to header string
    msg["Subject"] = f"Demolitoin Daily — {today_date}"

    msg.attach(MIMEText(md_text, "plain"))
    msg.attach(MIMEText(html, "html"))

    with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
        server.login(email_sender, email_pass)
        server.send_message(msg)

    return print("Email successfully sent!")

send_markdown_as_email("news_digest.md", email_sender, email_pass, email_recipients)

## Extra Code

### More SQL way of inserting new articles to db

In [ ]:
from sqlalchemy import create_engine, text
from datetime import datetime
import pytz

engine = create_engine("sqlite:///articles.db")

eastern = pytz.timezone('America/New_York')
local_time = datetime.now(eastern).strftime('%Y-%m-%d %H:%M:%S')

with engine.begin() as conn:
    for _, row in final_output.iterrows():
        # Check if article already exists by link
        exists_df = pd.read_sql_query(
            text("SELECT 1 FROM news_articles WHERE link = :link"),
            conn,
            params={"link": row['link']})

        # Properly check if result is empty
        if not exists_df.empty:
            continue

        # Insert new article
        conn.execute(
            text("""
                INSERT INTO news_articles 
                (title, link, source, language, published_date, last_modified, keywords, summary, category, upload_date)
                VALUES (:title, :link, :source, :language, :published_date, :last_modified, :keywords, :summary, :category, :upload_date)
            """),
            {
                "title": row["title"],
                "link": row["link"],
                "source": row["source"],
                "language": row["language"],
                "published_date": row["published_date"],
                "last_modified": row["last_modified"],
                "keywords": row["keywords"],
                "summary": row["summary"],
                "category": row["category"],
                "upload_date": local_time
            },
        )

        print(f"✅ Inserted new article: {row['title']}")

### Langchain LLM try

In [ ]:
import pandas as pd
from langchain_ollama import ChatOllama
from langchain.messages import SystemMessage, HumanMessage 

final_output = pd.read_excel("final_news_output.xlsx")

# Loading in example markdown file as a style guide
with open("template.md", "r") as f:
    markdown_example = f.read()

# Convert DataFrame to a list of dicts, better reasoning for llm & loopings
articles = final_output.to_dict(orient="records")

# Initialize your local LLM
llm = ChatOllama(model="qwen3:latest") 

# Building out system prompt
system_prompt = f"""
You are a professional technical editor who writes weekly news digests in Markdown format.

You'll be given a markdown template to mimic that will news stories bucketed under 5 categories. Each news story will belong to 
one of those cateogies and contains 3 outputs: the artcile title, artcile link, and article summary. You will be given an article in a dictionary 
format such as:

{'title': 'US airlines cancel more than 2,500 weekend flights amid shutdown',
  'link': 'https://apnews.com/article/flight-cancellations-government-shutdown-what-to-know-dea56ba97690b43397a96289f5f949e8',
  'source': 'Associated Press',
  'language': 'eng',
  'published_date': '2025-11-08T00:02:54-05:00',
  'last_modified': '2025-11-08T23:19:54-05:00',
  'keywords': None,
  'category': 'Politics',
  'summary': nan}

yYour job is to position the article in the correct markdown format in the correct news category.

Here is the example markdown file to mimic:

{markdown_example}
"""

# Building out human prompt
human_prompt = f"""
Here are the articles you need to format:

{articles}

Generate a cohesive Markdown file with headings, subheadings, and lists of articles.
Only return valid Markdown.
"""

# ✅ Get the model output
response = llm.invoke([
    SystemMessage(content=system_prompt),
    HumanMessage(content=human_prompt)
])

# ✅ Save the final markdown output
with open("weekly_digest.md", "w", encoding="utf-8") as f:
    f.write(response.content)

print("✅ Markdown file generated successfully: weekly_digest.md")


### Traditional web scrapping

In [ ]:
import requests
from bs4 import BeautifulSoup

url = 'https://apnews.com/hub/china'

html = requests.get(url)

soup = BeautifulSoup(html.content, 'html.parser')

In [ ]:
article_list = soup.find_all("div", class_='PageList-items-item') 

In [ ]:
for artcile in article_list:
    title = artcile.find("h1", class_="Headline")
    link = artcile.find("a")['href']
    print(f"Title: {title}, Link: {link}")

In [ ]:
lins

In [ ]:
article_list